# Batch analysis

Looks over finished batches. Reads only the report written by `hpc finalize` (`load` raises if it is missing); nothing here simulates.
The same figures as files: `uv run python -m moran_graphs.plots <batch>`.

Mean times are conditional on fixation. `rho` counts censored runs (hit `max_steps`) as not fixed; the verification report says if there were any.

In [ ]:
%load_ext autoreload
%autoreload 2  

import json

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import LogNorm

from moran_graphs.plots import (COLORS, LABELS, PROPERTIES, hover_scatter, lines, load, print_batch,
                                print_run_report, property_grid, scatter, sweep, time_distributions,
                                versus_random)

## Configuration

The only cell to edit. Several batches are stitched with a `batch` column; a graph in two batches stays two rows.

In [ ]:
BATCHES = ["2026-10-14-respiratory-size-sweep-2"]
R_VALUES = [1.1]  # the per-r figures need exactly one
DIRECTED = [False, True]  # [False] undirected only, [True] directed only

## What ran

Plan, submission, graphs built, then the verification report of each batch.

In [ ]:
for batch in BATCHES:
    print_batch(batch)

### How the batch went

Your wait from submission to the finished report, CPU time and speed, how much worker time went to waste, and what each r cost.

In [ ]:
for batch in BATCHES:
    print_run_report(batch)

## Load

In [ ]:
results, hists = load(BATCHES)
print(f"{results['hash'].nunique()} graphs, r values {sorted(results['r'].unique().tolist())}")
# Random graphs stay whatever DIRECTED is: they are the undirected cloud every type is compared to.
keep = results["r"].isin(R_VALUES) & (results["is_directed"].isin(DIRECTED) | (results["category"] == "Random"))
results = results[keep]
varying = [p for p in PROPERTIES if results[p].dtype != bool and results[p].nunique() > 1]
results.groupby(["category", "is_directed", "r"])[["rho", "steps_mean_given_fixed", "n_censored"]].agg(["count", "mean", "min", "max"])

In [ ]:
results.columns

In [ ]:
results[["batch", "name", "category", "n_nodes", "n_edges", "rho", "rho_lo", "rho_hi",
      "rho_minus_random", "steps_mean_given_fixed", "steps_se_given_fixed",
      "steps_mean_given_fixed_minus_random", "n_random_matches", "n_censored"]].sort_values("rho", ascending=False).head()

## Fixation probability and time

Hover a point to see its graph and recipe.

In [ ]:
results.shape == results[results["steps_mean_given_fixed"] > 0].shape

In [ ]:
n_runs = results["n_fixed"] + results["n_extinct"]
results["steps_mean_absorption"] = (results["n_fixed"] * results["steps_mean_given_fixed"] + results["n_extinct"] * results["steps_mean_given_extinct"]) / n_runs
results["steps_mean_absorption"]

In [ ]:
# Log x: mean times span 154 to 6e7 steps here, so on a linear axis almost every graph sits at 0.
hover_scatter(results, x="steps_mean_given_fixed", y="rho", marker_by="r").update_xaxes(type="log")

In [ ]:
hover_scatter(results, "steps_mean_given_fixed", "steps_sd_given_fixed")

In [ ]:
if results["r"].nunique() > 1:
    hover_scatter(results, "r", "rho").show()

## Against the random cloud, across sizes

Each graph type as a curve over N, against the random graphs with the same N and E (grey: median and interquartile band, 100 per size). Right panel: random mean time over the graph's mean time, so above 1 the graph fixes faster than random; the band is the random IQR over the graph's single value.
A type needs a cloud at every size and one graph per size, the rest is printed. The complete graph has no cloud (no other graph has its N and E).
The random graphs are undirected, so for directed types the cloud matches N and E but not the edge directions. Directed types with a single source (line, star, mammalian, fish) have rho = 1/N by construction.

In [ ]:
for directed in DIRECTED:
    versus_random(results, directed)

## Fixation time distributions

Runs that fixed, pooled over the graphs of each category, from the batch's step histograms.

In [ ]:
time_distributions(results, hists).axes[0].set_title(f"r = {R_VALUES}");

## Structural properties

One panel per property. Look for properties where the respiratory graphs sit outside the random cloud.

In [ ]:
property_grid(results, "rho", varying).suptitle(f"r = {R_VALUES}");

In [ ]:
property_grid(results, "steps_mean_given_fixed", varying).suptitle(f"r = {R_VALUES}");

## Parameter sweep

One category against a kwarg of its recipe, e.g. the asymmetric lungs against `flow_ratio`. Points are colored by N, because `n_nodes` is only a target and the built tree's N moves with the sweep.
Grey bars: the random graphs with the same N and E (min to max), at the same x.
Bottom row: each graph minus the mean of those random graphs (`*_minus_random` columns, added by `load`), which takes N out of the comparison. No error bars there: the spread of the random graphs (grey) is the reference.

Several kwarg values can build the same graph; the duplicates are dropped, so each point is the first value that built its graph (see the `graphs` warning in the verification report).

Dashed line: the human lung, `flow_ratio` = 0.326. In 2026-10-06-asymmetric-lungs it builds the same tree as 0.32, so that point (N = 33) is the human tree.

In [ ]:
SWEEP_CATEGORY, SWEEP_KWARG = "Asymmetric lung", "flow_ratio"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023, the asymmetric_lung_graph default

fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), layout="constrained", sharex=True)
ys = ("rho", "steps_mean_given_fixed", "rho_minus_random", "steps_mean_given_fixed_minus_random")
for ax, y in zip(axes.flat, ys):
    points = sweep(ax, results, SWEEP_CATEGORY, SWEEP_KWARG, y)
    if y.endswith("_minus_random"):
        ax.axhline(0, color=COLORS["Random"], lw=1)
    ax.axvline(HUMAN_FLOW_RATIO, color="0.2", ls="--", lw=1, label=f"human lung ({HUMAN_FLOW_RATIO})")
fig.colorbar(points, ax=axes, label="N")
axes[0, 0].legend(frameon=False)
fig.suptitle(f"r = {R_VALUES}");

## Asymmetric lungs across sizes

For a batch that sweeps `flow_ratio` at several target sizes (`n_nodes` in the recipe), e.g. 2026-10-07-asymmetric-lungs-size-sweep.

- **Left:** fixation probability minus the mean of the random trees with the same N and E, one line per target N. Below 0 the lung suppresses selection compared with a random tree of its size, above 0 it amplifies.
- **Middle and right:** each line is one `flow_ratio` across sizes, over the cloud of random trees (grey). The x position is the built N, which misses the target by a few percent.
- Time is divided by N² so all sizes share one linear axis; the random trees are scaled the same way.

Error bars are 95% intervals (Wilson for `rho`, 1.96 SE for time); the left panel has none, see the matched-random section above.

In [ ]:
LUNG_CATEGORY = "Asymmetric lung"
HUMAN_FLOW_RATIO = 0.326  # Kundu & Panchagnula 2023

# Time over N^2 keeps all sizes on one linear axis; random trees get the same scaling.
sized = results.assign(time_over_N2=results["steps_mean_given_fixed"] / results["n_nodes"] ** 2)
lungs = sized[sized["category"] == LUNG_CATEGORY]
recipe = pd.json_normalize(lungs["kwargs"].map(json.loads).tolist()).set_index(lungs.index)
lungs = lungs.assign(flow_ratio=recipe["flow_ratio"], **{"target N": recipe["n_nodes"]})
random = sized[sized["category"] == "Random"]
human = lungs[lungs["flow_ratio"] == HUMAN_FLOW_RATIO].sort_values("n_nodes")

fig, (ax_diff, ax_rho, ax_time) = plt.subplots(1, 3, figsize=(18, 5.5), layout="constrained")

lines(ax_diff, lungs, "flow_ratio", "rho_minus_random", by="target N")
ax_diff.axhline(0, color=COLORS["Random"], lw=1)
ax_diff.axvline(HUMAN_FLOW_RATIO, color="0.1", ls="--", lw=1, label=f"human lung ({HUMAN_FLOW_RATIO})")
ax_diff.legend(frameon=False, fontsize=9)
ax_diff.set_title("Lungs minus random trees of the same N and E")

for ax, y in ((ax_rho, "rho"), (ax_time, "time_over_N2")):
    ax.scatter(random["n_nodes"], random[y], s=8, alpha=0.5, linewidths=0, color=COLORS["Random"],
               label="random trees")
    lines(ax, lungs, "n_nodes", y, by="flow_ratio", cmap="plasma")
    ax.plot(human["n_nodes"], human[y], color="0.1", ls="--", lw=1.5, zorder=4, label="human lung")
    ax.set_xscale("log")
ax_rho.set_title("Fixation probability against size")
ax_time.set(title="Fixation time against size", ylabel="mean fixation time / N², given fixation")
fig.legend(*ax_time.get_legend_handles_labels(), loc="outside right center", frameon=False, fontsize=9)
fig.suptitle(f"Asymmetric lungs across sizes, r = {R_VALUES}");

### Heatmaps over target N and flow_ratio

Same lungs as above, one cell per (target N, `flow_ratio`). A blank cell is a `flow_ratio` that built the same tree as a smaller one at that size, dropped as a duplicate.
Fixation time is on a log color scale because it grows by orders of magnitude with N.

In [ ]:
fig, (ax_rho, ax_time) = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")
for ax, y, norm in ((ax_rho, "rho", None), (ax_time, "steps_mean_given_fixed", LogNorm())):
    grid = lungs.pivot(index="target N", columns="flow_ratio", values=y)
    image = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis", norm=norm)
    ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=45)
    ax.set_yticks(range(grid.shape[0]), grid.index)
    ax.set(xlabel="flow_ratio", ylabel="target N")
    fig.colorbar(image, ax=ax, label=LABELS[y])
fig.suptitle(f"Asymmetric lungs, r = {R_VALUES}");

## Ad hoc

Any two columns of `results`, or a subset of it. `hover=[...]` picks the columns shown on hover (default N, E, r, batch, recipe). `scatter(ax, df, x, y)` draws the same thing on a matplotlib axis.

In [ ]:
hover_scatter(results, "degree_std", "average_shortest_path_length")